# Starting the Code with Bismillah

# Setting Things Up below

## Model Loading other prelimnary Functions

In [ ]:
# ============================================================================
# Setup and imports
# ============================================================================
!pip install transformers accelerate datasets -q

import torch
import os
import math
from transformers import LlamaForCausalLM, LlamaTokenizer
from datasets import load_dataset
from tqdm.auto import tqdm
import gc

MODEL_NAME = "meta-llama/Llama-2-7b-hf"
PRUNING_RATIO = 0.20
OUTPUT_DIR = "./llama2_structured_pruned"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Device: {DEVICE}")
print(f"Target: {PRUNING_RATIO*100}% structured FFN pruning")

In [2]:
# ============================================================================
# Perplexity function
# ============================================================================
def calculate_perplexity(model, tokenizer, max_samples=100):
    """Calculate perplexity on wikitext-2 test set"""
    dataset = load_dataset("wikitext", "wikitext-2-raw-v1", split="test")
    texts = [t for t in dataset["text"] if len(t) > 50][:max_samples]

    model.eval()
    device = next(model.parameters()).device

    total_loss = 0
    total_tokens = 0

    with torch.no_grad():
        for text in tqdm(texts, desc="Calculating perplexity"):
            inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=512)
            inputs = {k: v.to(device) for k, v in inputs.items()}

            if inputs["input_ids"].shape[1] < 2:
                continue

            outputs = model(**inputs, labels=inputs["input_ids"])
            loss = outputs.loss

            num_tokens = inputs["input_ids"].shape[1] - 1
            total_loss += loss.item() * num_tokens
            total_tokens += num_tokens

    avg_loss = total_loss / total_tokens
    perplexity = math.exp(avg_loss)
    return perplexity

In [ ]:
# ============================================================================
# Load model and calculate BASELINE perplexity
# ============================================================================
print("Loading model...")
model = LlamaForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto"
)
tokenizer = LlamaTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token

num_layers = model.config.num_hidden_layers
intermediate_size = model.config.intermediate_size

print(f"Original FFN size: {intermediate_size}")
print(f"Original params: {sum(p.numel() for p in model.parameters()):,}")

print("\n" + "="*60)
print("Calculating BASELINE perplexity")
print("="*60)
baseline_ppl = calculate_perplexity(model, tokenizer, max_samples=100)
print(f"\n✓ Baseline Perplexity: {baseline_ppl:.2f}")

In [ ]:
# ============================================================================
# Load calibration data
# ============================================================================
print("Loading calibration data...")
dataset = load_dataset("wikitext", "wikitext-2-raw-v1", split="train")
calib_texts = [t for t in dataset["text"] if len(t) > 100][:128]
print(f"Calibration samples: {len(calib_texts)}")

Loading calibration data...
Calibration samples: 128


## Function to calculate Neuron Scores, mainly for Structured Pruning

### Target Selection: Feed-Forward Networks


In [ ]:
# ============================================================================
# Compute neuron importance
# ============================================================================
print("Measuring neuron activation importance...")

neuron_activations = {i: torch.zeros(intermediate_size, device=DEVICE) for i in range(num_layers)}
activation_counts = {i: 0 for i in range(num_layers)}

def make_activation_hook(layer_idx):
    def hook(module, input, output):
        act = output.detach().abs()
        neuron_activations[layer_idx] += act.sum(dim=[0, 1])
        activation_counts[layer_idx] += act.shape[0] * act.shape[1]
    return hook

hooks = []
for i, layer in enumerate(model.model.layers):
    h = layer.mlp.gate_proj.register_forward_hook(make_activation_hook(i))
    hooks.append(h)

model.eval()
with torch.no_grad():
    for text in tqdm(calib_texts, desc="Calibrating"):
        inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=256)
        inputs = {k: v.to(DEVICE) for k, v in inputs.items()}
        try:
            _ = model(**inputs)
        except:
            continue

for h in hooks:
    h.remove()

for i in range(num_layers):
    if activation_counts[i] > 0:
        neuron_activations[i] /= activation_counts[i]

print("✓ Neuron importance computed")

Measuring neuron activation importance...


Calibrating:   0%|          | 0/128 [00:00<?, ?it/s]

✓ Neuron importance computed


In [ ]:
# ============================================================================
# Select neurons to keep
# ============================================================================
new_intermediate_size = int(intermediate_size * (1 - PRUNING_RATIO))
new_intermediate_size = (new_intermediate_size // 128) * 128

print(f"New FFN size: {new_intermediate_size} (pruning {intermediate_size - new_intermediate_size} neurons/layer)")

neurons_to_keep = {}
for layer_idx in range(num_layers):
    importance = neuron_activations[layer_idx]
    _, top_indices = torch.topk(importance, new_intermediate_size)
    top_indices = top_indices.sort().values
    neurons_to_keep[layer_idx] = top_indices.cpu()  # Move to CPU for later use

del neuron_activations, activation_counts
gc.collect()
torch.cuda.empty_cache()

New FFN size: 8704 (pruning 2304 neurons/layer)


In [ ]:
# ============================================================================
# Delete old model, reload fresh, then prune
# ============================================================================
# We need to reload without accelerate hooks to avoid shape mismatch

print("Reloading model without accelerate hooks for pruning...")

# Delete old model
del model
gc.collect()
torch.cuda.empty_cache()

# Reload on CPU first (no device_map)
model = LlamaForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    low_cpu_mem_usage=True
)

print("Applying structured pruning...")

for layer_idx in tqdm(range(num_layers), desc="Pruning"):
    layer = model.model.layers[layer_idx]
    keep_idx = neurons_to_keep[layer_idx]

    # gate_proj: remove rows
    gate_w = layer.mlp.gate_proj.weight.data
    layer.mlp.gate_proj.weight = torch.nn.Parameter(gate_w[keep_idx, :].clone().contiguous())
    layer.mlp.gate_proj.out_features = new_intermediate_size

    # up_proj: remove rows
    up_w = layer.mlp.up_proj.weight.data
    layer.mlp.up_proj.weight = torch.nn.Parameter(up_w[keep_idx, :].clone().contiguous())
    layer.mlp.up_proj.out_features = new_intermediate_size

    # down_proj: remove columns
    down_w = layer.mlp.down_proj.weight.data
    layer.mlp.down_proj.weight = torch.nn.Parameter(down_w[:, keep_idx].clone().contiguous())
    layer.mlp.down_proj.in_features = new_intermediate_size

# Update config
model.config.intermediate_size = new_intermediate_size

new_params = sum(p.numel() for p in model.parameters())
original_params = 6_738_415_616

print(f"\n✓ Pruning complete")
print(f"  Params: {original_params:,} → {new_params:,}")
print(f"  Compression: {100*(1-new_params/original_params):.1f}%")

del neurons_to_keep
gc.collect()

Reloading model without accelerate hooks for pruning...


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Applying structured pruning...


Pruning:   0%|          | 0/32 [00:00<?, ?it/s]


✓ Pruning complete
  Params: 6,738,415,616 → 5,832,445,952
  Compression: 13.4%


119

In [ ]:
# ============================================================================
# Save pruned model
# ============================================================================
print("Saving pruned model...")
os.makedirs(OUTPUT_DIR, exist_ok=True)
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"✓ Saved to {OUTPUT_DIR}")

# Free memory
del model
gc.collect()
torch.cuda.empty_cache()

Saving pruned model...
✓ Saved to ./llama2_structured_pruned


In [ ]:
import shutil
import os

source_dir = "/content/llama2_structured_pruned"   # your actual pruned model folder
target_dir = "/content/drive/MyDrive/LLM_Compression_Project_final_hopefully/llama2_pruned_models"

if not os.path.exists(target_dir):
    os.makedirs(target_dir)

shutil.copytree(source_dir, target_dir, dirs_exist_ok=True)

print("✅ All pruned model files uploaded!!")


✅ All pruned model files uploaded!!


## Testing the Initital and Pruned Model's Perplexity.

In [ ]:
# ============================================================================
# Reload pruned model and calculate perplexity
# ============================================================================
print("Loading pruned model...")
model = LlamaForCausalLM.from_pretrained(
    OUTPUT_DIR,
    torch_dtype=torch.float16,
    device_map="auto"
)

print("\n" + "="*60)
print("Calculating PRUNED perplexity")
print("="*60)
pruned_ppl = calculate_perplexity(model, tokenizer, max_samples=100)

print(f"\n" + "="*60)
print("PERPLEXITY COMPARISON")
print("="*60)
print(f"  Baseline (original):  {baseline_ppl:.2f}")
print(f"  After pruning:        {pruned_ppl:.2f}")
print(f"  Degradation:          +{pruned_ppl - baseline_ppl:.2f} ({100*(pruned_ppl/baseline_ppl - 1):.1f}%)")
print("="*60)

Loading pruned model...


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]


Calculating PRUNED perplexity


Calculating perplexity:   0%|          | 0/100 [00:00<?, ?it/s]


PERPLEXITY COMPARISON
  Baseline (original):  8.47
  After pruning:        11.40
  Degradation:          +2.93 (34.5%)


In [ ]:
# ============================================================================
# Test inference
# ============================================================================
print("\nTesting inference...")

model.eval()
prompts = [
    "The capital of France is",
    "Machine learning is",
    "The largest planet in our solar system is"
]

for prompt in prompts:
    inputs = tokenizer(prompt, return_tensors="pt").to(DEVICE)

    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=40,
            do_sample=True,
            temperature=0.7,
            pad_token_id=tokenizer.eos_token_id
        )

    text = tokenizer.decode(out[0], skip_special_tokens=True)
    print(f"\n[Prompt]: {prompt}")
    print(f"[Output]: {text}")


Testing inference...

[Prompt]: The capital of France is
[Output]: The capital of France is Paris. It has been called the “city of love”, the “city of light”, and the “city of dreams”. It is the most popular tourist destination in the world.
Par

[Prompt]: Machine learning is
[Output]: Machine learning is a technique that is utilized by the developers to improve the performance of the applications. The developers have the ability to improve the performance of the applications by using the techniques. Machine learning is the technique that

[Prompt]: The largest planet in our solar system is
[Output]: The largest planet in our solar system is about 10 times the size of Earth. That's not a small planet, but a big one!
And the biggest planet in our solar system is about 1,000


In [ ]:
# ============================================================================
# Final summary
# ============================================================================
print("\n" + "="*60)
print("FINAL SUMMARY")
print("="*60)
print(f"  Pruning ratio:     {PRUNING_RATIO*100}%")
print(f"  FFN size:          {intermediate_size} → {new_intermediate_size}")
print(f"  Parameters:        {original_params:,} → {new_params:,}")
print(f"  Param reduction:   {100*(1-new_params/original_params):.1f}%")
print(f"  Baseline PPL:      {baseline_ppl:.2f}")
print(f"  Pruned PPL:        {pruned_ppl:.2f}")
print(f"  PPL increase:      {100*(pruned_ppl/baseline_ppl - 1):.1f}%")
print(f"  Model saved to:    {OUTPUT_DIR}")
print("="*60)


FINAL SUMMARY
  Pruning ratio:     20.0%
  FFN size:          11008 → 8704
  Parameters:        6,738,415,616 → 5,832,445,952
  Param reduction:   13.4%
  Baseline PPL:      8.47
  Pruned PPL:        11.40
  PPL increase:      34.5%
  Model saved to:    ./llama2_structured_pruned


# Here comes the Lllama.cpp

In [ ]:
# ============================================================================
# Install dependencies and build llama.cpp with CMake
# ============================================================================
!pip install safetensors -q

# Clone llama.cpp (skip if already cloned)
import os
if not os.path.exists("llama.cpp"):
    !git clone https://github.com/ggerganov/llama.cpp

# Build with CMake
%cd llama.cpp
!cmake -B build
!cmake --build build --config Release -j4

# Go back to root
%cd ..

# Verify build
!ls llama.cpp/build/bin/

Cloning into 'llama.cpp'...
remote: Enumerating objects: 72890, done.
remote: Counting objects: 100% (280/280), done.
remote: Compressing objects: 100% (202/202), done.
remote: Total 72890 (delta 183), reused 78 (delta 78), pack-reused 72610 (from 3)
Receiving objects: 100% (72890/72890), 249.83 MiB | 38.44 MiB/s, done.
Resolving deltas: 100% (52636/52636), done.
/content/llama.cpp
-- The C compiler identification is GNU 11.4.0
-- The CXX compiler identification is GNU 11.4.0
-- Detecting C compiler ABI info
-- Detecting C compiler ABI info - done
-- Check for working C compiler: /usr/bin/cc - skipped
-- Detecting C compile features
-- Detecting C compile features - done
-- Detecting CXX compiler ABI info
-- Detecting CXX compiler ABI info - done
-- Check for working CXX compiler: /usr/bin/c++ - skipped
-- Detecting CXX compile features
-- Detecting CXX compile features - done
CMAKE_BUILD_TYPE=Release
-- Found Git: /usr/bin/git (found version "2.34.1")
-- The ASM compiler identificatio

## ...and Quantization

In [ ]:
# ============================================================================
# Quantize PRUNED-ONLY Model (No SmoothQuant)
# ============================================================================

# Step 1: Convert pruned model to GGUF (if not already done)
print("Converting Pruned Model to GGUF F16...")
!python llama.cpp/convert_hf_to_gguf.py ./llama2_structured_pruned \
    --outfile ./llama2_pruned_f16.gguf --outtype f16

# Step 2: Quantize pruned to Q8_0 and Q4_0
print("\nQuantizing Pruned to Q8_0...")
!./llama.cpp/build/bin/llama-quantize ./llama2_pruned_f16.gguf \
    ./llama2_pruned_q8_0.gguf q8_0

print("\nQuantizing Pruned to Q4_0...")
!./llama.cpp/build/bin/llama-quantize ./llama2_pruned_f16.gguf \
    ./llama2_pruned_q4_0.gguf q4_0

print("\nQuantizing Pruned to Q4_K_M...")
!./llama.cpp/build/bin/llama-quantize ./llama2_pruned_f16.gguf \
    ./llama2_pruned_q4_k_m.gguf q4_k_m

Converting Pruned Model to GGUF F16...
INFO:hf-to-gguf:Loading model: llama2_structured_pruned
INFO:hf-to-gguf:Model architecture: LlamaForCausalLM
INFO:hf-to-gguf:gguf: loading model weight map from 'model.safetensors.index.json'
INFO:hf-to-gguf:gguf: indexing model part 'model-00001-of-00003.safetensors'
INFO:hf-to-gguf:gguf: indexing model part 'model-00002-of-00003.safetensors'
INFO:hf-to-gguf:gguf: indexing model part 'model-00003-of-00003.safetensors'
INFO:gguf.gguf_writer:gguf: This GGUF file is for Little Endian only
INFO:hf-to-gguf:Exporting model...
INFO:hf-to-gguf:token_embd.weight,           torch.float16 --> F16, shape = {4096, 32000}
INFO:hf-to-gguf:blk.0.attn_norm.weight,      torch.float16 --> F32, shape = {4096}
INFO:hf-to-gguf:blk.0.ffn_down.weight,       torch.float16 --> F16, shape = {8704, 4096}
INFO:hf-to-gguf:blk.0.ffn_gate.weight,       torch.float16 --> F16, shape = {4096, 8704}
INFO:hf-to-gguf:blk.0.ffn_up.weight,         torch.float16 --> F16, shape = {4096, 

## Smooth quant

In [ ]:

PRUNED_MODEL_PATH = "./llama2_structured_pruned"
OUTPUT_PATH = "./llama2_smoothquant_fixed"
NUM_CALIBRATION_SAMPLES = 64
ALPHA = 0.5

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

# ============================================================================
# FIXED SMOOTHQUANT, Load model
# ============================================================================

print("Loading pruned model...")
model = AutoModelForCausalLM.from_pretrained(
    PRUNED_MODEL_PATH,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True
)
tokenizer = AutoTokenizer.from_pretrained(PRUNED_MODEL_PATH)
model.eval()

num_layers = len(model.model.layers)
hidden_size = model.config.hidden_size
print(f"✓ Loaded model with {num_layers} layers, hidden_size={hidden_size}")

# ============================================================================
# FIXED SMOOTHQUANT, Load calibration data
# ============================================================================

print("Loading calibration data...")
dataset = load_dataset("wikitext", "wikitext-2-raw-v1", split="train")
calibration_texts = [t for t in dataset["text"] if len(t) > 100][:NUM_CALIBRATION_SAMPLES]

calibration_inputs = []
for text in calibration_texts:
    tokens = tokenizer(text, return_tensors="pt", truncation=True, max_length=512)
    calibration_inputs.append(tokens.input_ids)

print(f"✓ Prepared {len(calibration_inputs)} calibration samples")


`torch_dtype` is deprecated! Use `dtype` instead!


Device: cuda
Loading pruned model...


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

✓ Loaded model with 32 layers, hidden_size=4096
Loading calibration data...
✓ Prepared 64 calibration samples


In [ ]:
# ============================================================================
# FIXED SMOOTHQUANT, Collect activation statistics
# ============================================================================

print("=" * 70)
print("Collecting Activation Statistics")
print("=" * 70)

activation_abs_max = {}

def make_input_hook(name):
    def hook(module, input, output):
        inp = input[0].detach().float()
        max_per_channel = inp.view(-1, inp.shape[-1]).abs().max(dim=0)[0]
        if name not in activation_abs_max:
            activation_abs_max[name] = max_per_channel
        else:
            activation_abs_max[name] = torch.max(activation_abs_max[name], max_per_channel)
    return hook

hooks = []
for i, layer in enumerate(model.model.layers):
    hooks.append(layer.self_attn.q_proj.register_forward_hook(make_input_hook(f"layer.{i}.attn")))
    hooks.append(layer.mlp.gate_proj.register_forward_hook(make_input_hook(f"layer.{i}.ffn")))

print(f"Registered {len(hooks)} hooks")

print("Running calibration forward passes...")
with torch.no_grad():
    for inp in tqdm(calibration_inputs, desc="Calibration"):
        try:
            model(inp.to(device))
        except:
            continue

for hook in hooks:
    hook.remove()

print(f"✓ Collected activation stats for {len(activation_abs_max)} layer groups")

# ============================================================================
# FIXED SMOOTHQUANT, Apply SmoothQuant
# ============================================================================

print("=" * 70)
print("Applying SmoothQuant (Fixed: Scaling BOTH LayerNorm AND Linear)")
print("=" * 70)

def compute_smooth_factor(act_max, weight, alpha=0.5, eps=1e-5):
    weight_max = weight.abs().max(dim=0)[0].float()
    act_max = act_max.float().to(weight.device)
    s = (act_max.pow(alpha) / (weight_max.pow(1 - alpha) + eps))
    s = s.clamp(min=1e-5, max=1e5)
    return s

smoothed_count = 0

for i, layer in enumerate(tqdm(model.model.layers, desc="Applying SmoothQuant")):

    # Attention: input_layernorm -> q, k, v
    attn_key = f"layer.{i}.attn"
    if attn_key in activation_abs_max:
        act_max = activation_abs_max[attn_key]
        q_weight = layer.self_attn.q_proj.weight.data
        s_attn = compute_smooth_factor(act_max, q_weight, ALPHA)

        # KEY FIX: Scale LayerNorm weight by s^-1
        norm_weight = layer.input_layernorm.weight.data.float()
        layer.input_layernorm.weight.data = (norm_weight / s_attn).half()

        # Scale q, k, v projection weights by s
        for proj in [layer.self_attn.q_proj, layer.self_attn.k_proj, layer.self_attn.v_proj]:
            w = proj.weight.data.float()
            proj.weight.data = (w * s_attn.unsqueeze(0)).half()

        smoothed_count += 1

    # FFN: post_attention_layernorm -> gate, up
    ffn_key = f"layer.{i}.ffn"
    if ffn_key in activation_abs_max:
        act_max = activation_abs_max[ffn_key]
        gate_weight = layer.mlp.gate_proj.weight.data
        s_ffn = compute_smooth_factor(act_max, gate_weight, ALPHA)

        # KEY FIX: Scale LayerNorm weight by s^-1
        norm_weight = layer.post_attention_layernorm.weight.data.float()
        layer.post_attention_layernorm.weight.data = (norm_weight / s_ffn).half()

        # Scale gate and up projection weights by s
        for proj in [layer.mlp.gate_proj, layer.mlp.up_proj]:
            w = proj.weight.data.float()
            proj.weight.data = (w * s_ffn.unsqueeze(0)).half()

        smoothed_count += 1

print(f"✓ Applied SmoothQuant to {smoothed_count} layer groups")

# ============================================================================
# FIXED SMOOTHQUANT, Verify model works
# ============================================================================

print("=" * 70)
print("Verification: Testing Model Output")
print("=" * 70)

test_prompts = [
    "The capital of France is",
    "The largest planet in our solar system is",
    "Artificial intelligence is"
]

model.eval()
for prompt in test_prompts:
    inputs = tokenizer(prompt, return_tensors="pt").to(device)

    with torch.no_grad():
        outputs = model.generate(
            inputs.input_ids,
            max_new_tokens=30,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    generated = tokenizer.decode(outputs[0], skip_special_tokens=True)
    print(f"\nPrompt: {prompt}")
    print(f"Output: {generated}")

# ============================================================================
# FIXED SMOOTHQUANT, Save model
# ============================================================================

print("=" * 70)
print("Saving Fixed SmoothQuant Model")
print("=" * 70)

os.makedirs(OUTPUT_PATH, exist_ok=True)
model.save_pretrained(OUTPUT_PATH)
tokenizer.save_pretrained(OUTPUT_PATH)
print(f"✓ Model saved to {OUTPUT_PATH}")

del activation_abs_max
gc.collect()
torch.cuda.empty_cache()


Registered 64 hooks
Running calibration forward passes...


Calibration: 100%|██████████| 64/64 [00:28<00:00,  2.28it/s]


✓ Collected activation stats for 64 layer groups
Applying SmoothQuant (Fixed: Scaling BOTH LayerNorm AND Linear)


Applying SmoothQuant: 100%|██████████| 32/32 [00:00<00:00, 310.90it/s]


✓ Applied SmoothQuant to 64 layer groups
Verification: Testing Model Output


The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.



Prompt: The capital of France is
Output: The capital of France is a city of great history and culture. The city is home to many of the world’s most famous landmarks, including the Eiffel Tower

Prompt: The largest planet in our solar system is
Output: The largest planet in our solar system is Jupiter. It is the only planet in the solar system that has a glowing atmosphere. It is also the only planet in the solar system

Prompt: Artificial intelligence is
Output: Artificial intelligence is a field that has been around for a long time, but it has only recently started to gain traction.
The field of artificial intelligence (AI
Saving Fixed SmoothQuant Model


/usr/local/lib/python3.12/dist-packages/transformers/modeling_utils.py:3970: UserWarning: Attempting to save a model with offloaded modules. Ensure that unallocated cpu memory exceeds the `shard_size` (5GB default)
  warnings.warn(


Saving checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

✓ Model saved to ./llama2_smoothquant_fixed


# Mixed precision with custom built program

In [ ]:
# ============================================================================
# Mixed Precision, Custom Sensitivity Analysis
# ============================================================================

SMOOTHQUANT_MODEL_PATH = "./llama2_smoothquant_fixed"
OUTPUT_CONFIG_PATH = "./mixed_precision_config.json"

print("=" * 70)
print("MIXED PRECISION: Custom Sensitivity Analysis")
print("=" * 70)

# Load model
model = AutoModelForCausalLM.from_pretrained(
    SMOOTHQUANT_MODEL_PATH,
    torch_dtype=torch.float16,
    device_map="auto"
)
tokenizer = AutoTokenizer.from_pretrained(SMOOTHQUANT_MODEL_PATH)

# Load calibration data
dataset = load_dataset("wikitext", "wikitext-2-raw-v1", split="test")
calibration_texts = [t for t in dataset["text"] if len(t) > 100][:32]

calibration_inputs = []
for text in calibration_texts:
    tokens = tokenizer(text, return_tensors="pt", truncation=True, max_length=512)
    calibration_inputs.append(tokens.input_ids)


MIXED PRECISION: Custom Sensitivity Analysis


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

In [ ]:

# ============================================================================
# Weight-based Sensitivity
# ============================================================================
print("\n[Step 1] Computing Weight-based Sensitivity...")

weight_sensitivity = {}
for name, module in model.named_modules():
    if isinstance(module, nn.Linear):
        weight = module.weight.data.float()
        weight_std = weight.std().item()
        weight_outlier_ratio = (weight.abs() > 3 * weight.std()).float().mean().item()
        sensitivity = weight_std * (1 + weight_outlier_ratio * 10)
        weight_sensitivity[name] = sensitivity

# ============================================================================
# Activation-based Sensitivity
# ============================================================================
print("[Step 2] Computing Activation-based Sensitivity...")

activation_stats = {}

def make_hook(name):
    def hook(module, input, output):
        out = output[0] if isinstance(output, tuple) else output
        out_flat = out.detach().float().view(-1)
        if name not in activation_stats:
            activation_stats[name] = {'sum': 0, 'sum_sq': 0, 'count': 0, 'max': 0}
        activation_stats[name]['sum'] += out_flat.sum().item()
        activation_stats[name]['sum_sq'] += (out_flat ** 2).sum().item()
        activation_stats[name]['count'] += out_flat.numel()
        activation_stats[name]['max'] = max(activation_stats[name]['max'], out_flat.abs().max().item())
    return hook

hooks = []
for name, module in model.named_modules():
    if isinstance(module, nn.Linear):
        hooks.append(module.register_forward_hook(make_hook(name)))

model.eval()
with torch.no_grad():
    for inp in tqdm(calibration_inputs[:16], desc="Calibration"):
        try:
            model(inp.to(model.device))
        except:
            continue

for hook in hooks:
    hook.remove()

activation_sensitivity = {}
for name, stats in activation_stats.items():
    if stats['count'] > 0:
        mean = stats['sum'] / stats['count']
        var = (stats['sum_sq'] / stats['count']) - (mean ** 2)
        std = var ** 0.5 if var > 0 else 0
        activation_sensitivity[name] = std * (1 + stats['max'] / 100)

# ============================================================================
# Combined Sensitivity
# ============================================================================
print("[Step 3] Computing Combined Sensitivity...")

combined_sensitivity = {}
for name in weight_sensitivity.keys():
    w_sens = weight_sensitivity[name]
    a_sens = activation_sensitivity.get(name, 0)
    combined_sensitivity[name] = 0.7 * w_sens + 0.3 * a_sens



[Step 1] Computing Weight-based Sensitivity...
[Step 2] Computing Activation-based Sensitivity...


Calibration: 100%|██████████| 16/16 [00:02<00:00,  7.11it/s]

[Step 3] Computing Combined Sensitivity...


In [ ]:

# ============================================================================
# Allocate Bit-widths
# ============================================================================
print("\n" + "=" * 70)
print("BIT-WIDTH ALLOCATION RESULTS")
print("=" * 70)

sorted_layers = sorted(combined_sensitivity.items(), key=lambda x: x[1], reverse=True)
num_layers = len(sorted_layers)
q8_count = int(num_layers * 0.25)

q8_layers = [name for name, _ in sorted_layers[:q8_count]]
q4_layers = [name for name, _ in sorted_layers[q8_count:]]

print(f"\nTotal Linear Layers: {num_layers}")
print(f"Q8 (8-bit): {len(q8_layers)} layers ({len(q8_layers)/num_layers*100:.1f}%) - Most sensitive")
print(f"Q4 (4-bit): {len(q4_layers)} layers ({len(q4_layers)/num_layers*100:.1f}%) - Least sensitive")

print("\n--- Top 10 Most Sensitive Layers (Q8) ---")
for name in q8_layers[:10]:
    print(f"  {name}: {combined_sensitivity[name]:.4f}")

print("\n--- Top 5 Least Sensitive Layers (Q4) ---")
for name in q4_layers[-5:]:
    print(f"  {name}: {combined_sensitivity[name]:.4f}")

# Save config
config = {
    'method': 'combined_weight_activation_sensitivity',
    'weight_ratio': 0.7,
    'activation_ratio': 0.3,
    'q8_ratio': 0.25,
    'q4_ratio': 0.75,
    'q8_layers': q8_layers,
    'q4_layers': q4_layers,
    'combined_sensitivity': combined_sensitivity
}

with open(OUTPUT_CONFIG_PATH, 'w') as f:
    json.dump(config, f, indent=2)

print(f"\n✓ Config saved to {OUTPUT_CONFIG_PATH}")

# Cleanup
del model
gc.collect()
torch.cuda.empty_cache()

print("\n" + "=" * 70)
print("NOTE: llama.cpp Q4_K_M uses similar sensitivity-based allocation.")
print("Our analysis validates this approach with custom metrics.")
print("=" * 70)


BIT-WIDTH ALLOCATION RESULTS

Total Linear Layers: 225
Q8 (8-bit): 56 layers (24.9%) - Most sensitive
Q4 (4-bit): 169 layers (75.1%) - Least sensitive

--- Top 10 Most Sensitive Layers (Q8) ---
  model.layers.30.mlp.down_proj: 49.9775
  model.layers.1.mlp.down_proj: 22.4641
  model.layers.31.mlp.down_proj: 1.7479
  lm_head: 1.4911
  model.layers.16.self_attn.k_proj: 0.6523
  model.layers.27.self_attn.k_proj: 0.6505
  model.layers.22.self_attn.k_proj: 0.6462
  model.layers.17.self_attn.k_proj: 0.6444
  model.layers.15.self_attn.k_proj: 0.6389
  model.layers.14.self_attn.k_proj: 0.6380

--- Top 5 Least Sensitive Layers (Q4) ---
  model.layers.0.mlp.down_proj: 0.0165
  model.layers.0.self_attn.v_proj: 0.0161
  model.layers.2.self_attn.o_proj: 0.0154
  model.layers.1.self_attn.o_proj: 0.0123
  model.layers.0.self_attn.o_proj: 0.0107

✓ Config saved to ./mixed_precision_config.json

NOTE: llama.cpp Q4_K_M uses similar sensitivity-based allocation.
Our analysis validates this approach with 

In [ ]:
# ============================================================================
# Convert to GGUF, Multiple Quantization Levels
# ============================================================================

import os

print("=" * 70)
print("GGUF CONVERSION")
print("=" * 70)

# Convert SmoothQuant model to GGUF F16
print("\n[1/4] Converting to GGUF F16...")
!python llama.cpp/convert_hf_to_gguf.py /content/llama2_smoothquant_fixed \
    --outfile ./llama2_smoothquant_fixed_f16.gguf --outtype f16

# Quantize to Q8_0
print("\n[2/4] Quantizing to Q8_0...")
!./llama.cpp/build/bin/llama-quantize ./llama2_smoothquant_fixed_f16.gguf \
    ./llama2_smoothquant_fixed_q8_0.gguf q8_0

# Quantize to Q4_K_M (mixed precision)
print("\n[3/4] Quantizing to Q4_K_M (mixed precision)...")
!./llama.cpp/build/bin/llama-quantize ./llama2_smoothquant_fixed_f16.gguf \
    ./llama2_smoothquant_fixed_q4_k_m.gguf q4_k_m

# Quantize to Q4_0
print("\n[4/4] Quantizing to Q4_0...")
!./llama.cpp/build/bin/llama-quantize ./llama2_smoothquant_fixed_f16.gguf \
    ./llama2_smoothquant_fixed_q4_0.gguf q4_0

# Check file sizes
print("\n" + "=" * 70)
print("GGUF FILE SIZES")
print("=" * 70)

gguf_files = [
    "llama2_smoothquant_fixed_f16.gguf",
    "llama2_smoothquant_fixed_q8_0.gguf",
    "llama2_smoothquant_fixed_q4_k_m.gguf",
    "llama2_smoothquant_fixed_q4_0.gguf"
]

for f in gguf_files:
    if os.path.exists(f"./{f}"):
        size_gb = os.path.getsize(f"./{f}") / (1024**3)
        print(f"  {f}: {size_gb:.2f} GB")
    else:
        print(f"  {f}: Not found")

print("\n✓ GGUF Conversion Complete!")

GGUF CONVERSION

[1/4] Converting to GGUF F16...
INFO:hf-to-gguf:Loading model: llama2_smoothquant_fixed
INFO:hf-to-gguf:Model architecture: LlamaForCausalLM
INFO:hf-to-gguf:gguf: loading model weight map from 'model.safetensors.index.json'
INFO:hf-to-gguf:gguf: indexing model part 'model-00001-of-00003.safetensors'
INFO:hf-to-gguf:gguf: indexing model part 'model-00002-of-00003.safetensors'
INFO:hf-to-gguf:gguf: indexing model part 'model-00003-of-00003.safetensors'
INFO:gguf.gguf_writer:gguf: This GGUF file is for Little Endian only
INFO:hf-to-gguf:Exporting model...
INFO:hf-to-gguf:token_embd.weight,           torch.float16 --> F16, shape = {4096, 32000}
INFO:hf-to-gguf:blk.0.attn_norm.weight,      torch.float16 --> F32, shape = {4096}
INFO:hf-to-gguf:blk.0.ffn_down.weight,       torch.float16 --> F16, shape = {8704, 4096}
INFO:hf-to-gguf:blk.0.ffn_gate.weight,       torch.float16 --> F16, shape = {4096, 8704}
INFO:hf-to-gguf:blk.0.ffn_up.weight,         torch.float16 --> F16, shape

In [ ]:
# ============================================================================
# FIXED SMOOTHQUANT, Convert to GGUF
# ============================================================================

print("=" * 70)
print("Converting to GGUF")
print("=" * 70)

!python llama.cpp/convert_hf_to_gguf.py ./llama2_smoothquant_fixed \
    --outfile ./llama2_smoothquant_fixed_f16.gguf --outtype f16

!./llama.cpp/build/bin/llama-quantize ./llama2_smoothquant_fixed_f16.gguf \
    ./llama2_smoothquant_fixed_q4_k_m.gguf q4_k_m

!./llama.cpp/build/bin/llama-quantize ./llama2_smoothquant_fixed_f16.gguf \
    ./llama2_smoothquant_fixed_q4_0.gguf q4_0

!./llama.cpp/build/bin/llama-quantize ./llama2_smoothquant_fixed_f16.gguf \
    ./llama2_smoothquant_fixed_q8_0.gguf q8_0

# Check sizes
print("\n" + "=" * 70)
print("GGUF FILE SIZES")
print("=" * 70)
for f in ["llama2_smoothquant_fixed_f16.gguf", "llama2_smoothquant_fixed_q8_0.gguf",
          "llama2_smoothquant_fixed_q4_k_m.gguf", "llama2_smoothquant_fixed_q4_0.gguf"]:
    if os.path.exists(f"./{f}"):
        size_gb = os.path.getsize(f"./{f}") / (1024**3)
        print(f"  {f}: {size_gb:.2f} GB")


Converting to GGUF
INFO:hf-to-gguf:Loading model: llama2_smoothquant_fixed
INFO:hf-to-gguf:Model architecture: LlamaForCausalLM
INFO:hf-to-gguf:gguf: loading model weight map from 'model.safetensors.index.json'
INFO:hf-to-gguf:gguf: indexing model part 'model-00001-of-00003.safetensors'
INFO:hf-to-gguf:gguf: indexing model part 'model-00002-of-00003.safetensors'
INFO:hf-to-gguf:gguf: indexing model part 'model-00003-of-00003.safetensors'
INFO:gguf.gguf_writer:gguf: This GGUF file is for Little Endian only
INFO:hf-to-gguf:Exporting model...
INFO:hf-to-gguf:token_embd.weight,           torch.float16 --> F16, shape = {4096, 32000}
INFO:hf-to-gguf:blk.0.attn_norm.weight,      torch.float16 --> F32, shape = {4096}
INFO:hf-to-gguf:blk.0.ffn_down.weight,       torch.float16 --> F16, shape = {8704, 4096}
INFO:hf-to-gguf:blk.0.ffn_gate.weight,       torch.float16 --> F16, shape = {4096, 8704}
INFO:hf-to-gguf:blk.0.ffn_up.weight,         torch.float16 --> F16, shape = {4096, 8704}
INFO:hf-to-ggu

## And the testing phase on different Models using llama.cpp-completion (just endure some useless ran cells :))

In [ ]:
!./llama.cpp/build/bin/llama-completion -m /content/drive/MyDrive/LLM_Compression_Project_final_hopefully/llama2_pruned_f16.gguf \
    -p "The largest planet in our solar system is" -n 40 --temp 0.7

build: 7489 (10b4f82d4) with GNU 11.4.0 for Linux x86_64
main: llama backend init
main: load the model and apply lora adapter, if any
common_init_result: fitting params to device memory, for bugs during this step try to reproduce them with -fit off, or provide --verbose logs if the bug only occurs with -fit on
llama_params_fit_impl: no devices with dedicated memory found
llama_params_fit: successfully fit params to free device memory
llama_params_fit: fitting params to free memory took 1.97 seconds
llama_model_loader: loaded meta data with 30 key-value pairs and 291 tensors from /content/drive/MyDrive/LLM_Compression_Project_final_hopefully/llama2_pruned_f16.gguf (version GGUF V3 (latest))
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = llama
llama_model_loader: - kv   1:                               general.type str              = model
llama_m

In [ ]:
# ============================================================================
# FIXED SMOOTHQUANT, Test GGUF inference
# ============================================================================

print("=" * 70)
print("Testing GGUF Inference")
print("=" * 70)

!./llama.cpp/build/bin/llama-completion -m ./llama2_pruned_f16.gguf \
    -p "The largest planet in our solar system is" -n 40 --temp 0.7


!./llama.cpp/build/bin/llama-completion -m /content/drive/MyDrive/LLM_Compression_Project_final_hopefully/llama2_pruned_q4_0.gguf \
    -p "The largest planet in our solar system is" -n 40 --temp 0.7


Testing GGUF Inference
build: 7489 (10b4f82d4) with GNU 11.4.0 for Linux x86_64
main: llama backend init
main: load the model and apply lora adapter, if any
common_init_result: fitting params to device memory, for bugs during this step try to reproduce them with -fit off, or provide --verbose logs if the bug only occurs with -fit on
llama_params_fit_impl: no devices with dedicated memory found
llama_params_fit: successfully fit params to free device memory
llama_params_fit: fitting params to free memory took 1.25 seconds
llama_model_loader: loaded meta data with 30 key-value pairs and 291 tensors from /content/drive/MyDrive/LLM_Compression_Project_final_hopefully/llama2_pruned_q4_0.gguf (version GGUF V3 (latest))
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = llama
llama_model_loader: - kv   1:                               general.type str     

In [ ]:
# ============================================================================
# FIXED SMOOTHQUANT, Test GGUF inference
# ============================================================================

print("=" * 70)
print("Testing GGUF Inference")
print("=" * 70)

!./llama.cpp/build/bin/llama-completion -m ./llama2_pruned_q4_0.gguf \
    -p "The largest planet in our solar system is" -n 40 --temp 0.7

print("\n")

Testing GGUF Inference
build: 7489 (10b4f82d4) with GNU 11.4.0 for Linux x86_64
main: llama backend init
main: load the model and apply lora adapter, if any
common_init_result: fitting params to device memory, for bugs during this step try to reproduce them with -fit off, or provide --verbose logs if the bug only occurs with -fit on
llama_params_fit_impl: no devices with dedicated memory found
llama_params_fit: successfully fit params to free device memory
llama_params_fit: fitting params to free memory took 0.07 seconds
llama_model_loader: loaded meta data with 30 key-value pairs and 291 tensors from ./llama2_pruned_q4_0.gguf (version GGUF V3 (latest))
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = llama
llama_model_loader: - kv   1:                               general.type str              = model
llama_model_loader: - kv   2:               

In [ ]:
# ============================================================================
# FIXED SMOOTHQUANT, Test GGUF inference
# ============================================================================

print("=" * 70)
print("Testing GGUF Inference")
print("=" * 70)

!./llama.cpp/build/bin/llama-completion -m ./llama2_pruned_q4_k_m.gguf \
    -p "The largest planet in our solar system is" -n 40 --temp 0.7

print("\n")

Testing GGUF Inference
build: 7489 (10b4f82d4) with GNU 11.4.0 for Linux x86_64
main: llama backend init
main: load the model and apply lora adapter, if any
common_init_result: fitting params to device memory, for bugs during this step try to reproduce them with -fit off, or provide --verbose logs if the bug only occurs with -fit on
llama_params_fit_impl: no devices with dedicated memory found
llama_params_fit: successfully fit params to free device memory
llama_params_fit: fitting params to free memory took 0.07 seconds
llama_model_loader: loaded meta data with 30 key-value pairs and 291 tensors from ./llama2_pruned_q4_k_m.gguf (version GGUF V3 (latest))
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = llama
llama_model_loader: - kv   1:                               general.type str              = model
llama_model_loader: - kv   2:             

In [ ]:
# ============================================================================
# FIXED SMOOTHQUANT, Test GGUF inference
# ============================================================================

print("=" * 70)
print("Testing GGUF Inference")
print("=" * 70)

!./llama.cpp/build/bin/llama-completion -m ./llama2_pruned_q8_0.gguf \
    -p "The largest planet in our solar system is" -n 40 --temp 0.7

print("\n")

Testing GGUF Inference
build: 7489 (10b4f82d4) with GNU 11.4.0 for Linux x86_64
main: llama backend init
main: load the model and apply lora adapter, if any
common_init_result: fitting params to device memory, for bugs during this step try to reproduce them with -fit off, or provide --verbose logs if the bug only occurs with -fit on
llama_params_fit_impl: no devices with dedicated memory found
llama_params_fit: successfully fit params to free device memory
llama_params_fit: fitting params to free memory took 0.07 seconds
llama_model_loader: loaded meta data with 30 key-value pairs and 291 tensors from ./llama2_pruned_q8_0.gguf (version GGUF V3 (latest))
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = llama
llama_model_loader: - kv   1:                               general.type str              = model
llama_model_loader: - kv   2:               

In [ ]:
# ============================================================================
# FIXED SMOOTHQUANT, Test GGUF inference
# ============================================================================

print("=" * 70)
print("Testing GGUF Inference")
print("=" * 70)

!./llama.cpp/build/bin/llama-completion -m ./llama2_smoothquant_fixed_f16.gguf \
    -p "The largest planet in our solar system is" -n 40 --temp 0.7

print("\n")

Testing GGUF Inference
build: 7489 (10b4f82d4) with GNU 11.4.0 for Linux x86_64
main: llama backend init
main: load the model and apply lora adapter, if any
common_init_result: fitting params to device memory, for bugs during this step try to reproduce them with -fit off, or provide --verbose logs if the bug only occurs with -fit on
llama_params_fit_impl: no devices with dedicated memory found
llama_params_fit: successfully fit params to free device memory
llama_params_fit: fitting params to free memory took 0.07 seconds
llama_model_loader: loaded meta data with 33 key-value pairs and 291 tensors from ./llama2_smoothquant_fixed_f16.gguf (version GGUF V3 (latest))
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = llama
llama_model_loader: - kv   1:                               general.type str              = model
llama_model_loader: - kv   2:     

In [ ]:
# ============================================================================
# FIXED SMOOTHQUANT, Test GGUF inference
# ============================================================================

print("=" * 70)
print("Testing GGUF Inference")
print("=" * 70)

!./llama.cpp/build/bin/llama-completion -m ./llama2_smoothquant_fixed_q4_0.gguf \
    -p "The largest planet in our solar system is" -n 40 --temp 0.7

print("\n")

Testing GGUF Inference
build: 7489 (10b4f82d4) with GNU 11.4.0 for Linux x86_64
main: llama backend init
main: load the model and apply lora adapter, if any
common_init_result: fitting params to device memory, for bugs during this step try to reproduce them with -fit off, or provide --verbose logs if the bug only occurs with -fit on
llama_params_fit_impl: no devices with dedicated memory found
llama_params_fit: successfully fit params to free device memory
llama_params_fit: fitting params to free memory took 0.07 seconds
llama_model_loader: loaded meta data with 33 key-value pairs and 291 tensors from ./llama2_smoothquant_fixed_q4_0.gguf (version GGUF V3 (latest))
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = llama
llama_model_loader: - kv   1:                               general.type str              = model
llama_model_loader: - kv   2:    

In [ ]:
# ============================================================================
# FIXED SMOOTHQUANT, Test GGUF inference
# ============================================================================

print("=" * 70)
print("Testing GGUF Inference")
print("=" * 70)

!./llama.cpp/build/bin/llama-completion -m ./llama2_smoothquant_fixed_q4_k_m.gguf \
    -p "The largest planet in our solar system is" -n 40 --temp 0.7

print("\n")

Testing GGUF Inference
build: 7489 (10b4f82d4) with GNU 11.4.0 for Linux x86_64
main: llama backend init
main: load the model and apply lora adapter, if any
common_init_result: fitting params to device memory, for bugs during this step try to reproduce them with -fit off, or provide --verbose logs if the bug only occurs with -fit on
llama_params_fit_impl: no devices with dedicated memory found
llama_params_fit: successfully fit params to free device memory
llama_params_fit: fitting params to free memory took 0.07 seconds
llama_model_loader: loaded meta data with 33 key-value pairs and 291 tensors from ./llama2_smoothquant_fixed_q4_k_m.gguf (version GGUF V3 (latest))
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = llama
llama_model_loader: - kv   1:                               general.type str              = model
llama_model_loader: - kv   2:  

In [ ]:
# ============================================================================
# FIXED SMOOTHQUANT, Test GGUF inference
# ============================================================================

print("=" * 70)
print("Testing GGUF Inference")
print("=" * 70)

!./llama.cpp/build/bin/llama-completion -m ./llama2_smoothquant_fixed_q8_0.gguf \
    -p "The largest planet in our solar system is" -n 40 --temp 0.7

print("\n")

Testing GGUF Inference
build: 7489 (10b4f82d4) with GNU 11.4.0 for Linux x86_64
main: llama backend init
main: load the model and apply lora adapter, if any
common_init_result: fitting params to device memory, for bugs during this step try to reproduce them with -fit off, or provide --verbose logs if the bug only occurs with -fit on
llama_params_fit_impl: no devices with dedicated memory found
llama_params_fit: successfully fit params to free device memory
llama_params_fit: fitting params to free memory took 0.07 seconds
llama_model_loader: loaded meta data with 33 key-value pairs and 291 tensors from ./llama2_smoothquant_fixed_q8_0.gguf (version GGUF V3 (latest))
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = llama
llama_model_loader: - kv   1:                               general.type str              = model
llama_model_loader: - kv   2:    

In [ ]:
# ============================================================================
# FIXED SMOOTHQUANT, Test GGUF inference
# ============================================================================

print("=" * 70)
print("Testing GGUF Inference")
print("=" * 70)

!./llama.cpp/build/bin/llama-completion -m ./llama2_smoothquant_fixed_q4_k_m.gguf \
    -p "The largest planet in our solar system is" -n 40 --temp 0.7

print("\n")

Testing GGUF Inference
build: 7489 (10b4f82d4) with GNU 11.4.0 for Linux x86_64
main: llama backend init
main: load the model and apply lora adapter, if any
common_init_result: fitting params to device memory, for bugs during this step try to reproduce them with -fit off, or provide --verbose logs if the bug only occurs with -fit on
llama_params_fit_impl: no devices with dedicated memory found
llama_params_fit: successfully fit params to free device memory
llama_params_fit: fitting params to free memory took 0.07 seconds
llama_model_loader: loaded meta data with 33 key-value pairs and 291 tensors from ./llama2_smoothquant_fixed_q4_k_m.gguf (version GGUF V3 (latest))
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = llama
llama_model_loader: - kv   1:                               general.type str              = model
llama_model_loader: - kv   2:  

## The most part of the code, saving the models on drive.

In [ ]:
# ============================================================================
# CELL: Save Models to Google Drive
# ============================================================================

from google.colab import drive
import shutil
import os

# Mount Google Drive
drive.mount('/content/drive')

# Create project folder
output_folder = '/content/drive/MyDrive/LLM_Compression_Project_final_hopefully' #was too tired to name the drive folder ;)
os.makedirs(output_folder, exist_ok=True)

# Files to save
files_to_copy = [
    # Fixed SmoothQuant models (MAIN DELIVERABLES)
    'llama2_smoothquant_fixed_f16.gguf',
    './llama2_smoothquant_fixed_q4_0.gguf',
    './llama2_smoothquant_fixed_q4_k_m.gguf',
    './llama2_smoothquant_fixed_q8_0.gguf'
]

print("Copying files to Google Drive...")
print("=" * 60)

for file_path in files_to_copy:
    if os.path.exists(file_path):
        filename = os.path.basename(file_path)
        dest_path = os.path.join(output_folder, filename)
        print(f"Copying {filename}...", end=" ")
        shutil.copy2(file_path, dest_path)
        size_gb = os.path.getsize(dest_path) / (1024**3)
        print(f"✓ ({size_gb:.2f} GB)")
    else:
        print(f"✗ Not found: {file_path}")

print("=" * 60)
print(f"✓ All files saved to: {output_folder}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Copying files to Google Drive...
Copying llama2_smoothquant_fixed_f16.gguf... ✓ (10.86 GB)
Copying llama2_smoothquant_fixed_q4_0.gguf... ✓ (3.09 GB)
Copying llama2_smoothquant_fixed_q4_k_m.gguf... ✓ (3.29 GB)
Copying llama2_smoothquant_fixed_q8_0.gguf... ✓ (5.77 GB)
✓ All files saved to: /content/drive/MyDrive/LLM_Compression_Project_final_hopefully


# Evaluation Metrics


In [ ]:
# ============================================================================
# COMPREHENSIVE PROFILING & EVALUATION OF ALL GGUF MODELS
# ============================================================================

import subprocess
import re
import os

print("=" * 90)
print("COMPREHENSIVE PROFILING & EVALUATION")
print("=" * 90)


COMPREHENSIVE PROFILING & EVALUATION


In [ ]:

# ============================================================================
# PART 1: Model Size Comparison
# ============================================================================

print("\n" + "=" * 70)
print("PART 1: MODEL SIZES")
print("=" * 70)

all_models = [
    # Pruned-only models
    ("Pruned F16", "/content/drive/MyDrive/LLM_Compression_Project_final_hopefully/llama2_pruned_f16.gguf"),
    ("Pruned Q8_0", "/content/drive/MyDrive/LLM_Compression_Project_final_hopefully/llama2_pruned_q8_0.gguf"),
    ("Pruned Q4_K_M", "/content/drive/MyDrive/LLM_Compression_Project_final_hopefully/llama2_pruned_q4_k_m.gguf"),
    ("Pruned Q4_0", "/content/drive/MyDrive/LLM_Compression_Project_final_hopefully/llama2_pruned_q4_0.gguf"),

    # Fixed SmoothQuant models
    ("SmoothQuant Fixed f16" , "/content/llama2_smoothquant_fixed_f16.gguf"),
    ("SmoothQuant Fixed Q8_0", "/content/llama2_smoothquant_fixed_q8_0.gguf"),
    ("SmoothQuant Fixed Q4_K_M", "/content/llama2_smoothquant_fixed_q4_k_m.gguf"),
    ("SmoothQuant Fixed Q4_0", "/content/llama2_smoothquant_fixed_q4_0.gguf"),
]

original_size = 13.50  # LLaMA-2-7B FP16 reference

print(f"\n{'Model':<30} {'Size (GB)':>12} {'Compression':>12}")
print("-" * 55)

model_sizes = {}
for name, path in all_models:
    if os.path.exists(path):
        size_gb = os.path.getsize(path) / (1024**3)
        compression = (1 - size_gb / original_size) * 100
        model_sizes[name] = size_gb
        print(f"{name:<30} {size_gb:>12.2f} {compression:>11.1f}%")
    else:
        print(f"{name:<30} {'Not found':>12}")



PART 1: MODEL SIZES

Model                             Size (GB)  Compression
-------------------------------------------------------
Pruned F16                            10.86        19.5%
Pruned Q8_0                            5.77        57.2%
Pruned Q4_K_M                          3.29        75.6%
Pruned Q4_0                            3.09        77.1%
SmoothQuant Fixed f16                 10.86        19.5%
SmoothQuant Fixed Q8_0                 5.77        57.2%
SmoothQuant Fixed Q4_K_M               3.29        75.6%
SmoothQuant Fixed Q4_0                 3.09        77.1%


In [ ]:

# ============================================================================
# PART 2: Profiling Function
# ============================================================================

def profile_model(model_path, prompt="The quick brown fox jumps over the lazy dog", n_tokens=50):
    """Profile a GGUF model and extract all metrics"""

    if not os.path.exists(model_path):
        return None

    size_gb = os.path.getsize(model_path) / (1024**3)

    cmd = [
        "./llama.cpp/build/bin/llama-completion",
        "-m", model_path,
        "-p", prompt,
        "-n", str(n_tokens),
        "--temp", "0.7"
    ]

    try:
        result = subprocess.run(cmd, capture_output=True, text=True, timeout=300)
        output = result.stderr + result.stdout

        # Extract TTFT (prompt eval time)
        ttft_match = re.search(r'prompt eval time\s*=\s*([\d.]+)\s*ms', output)
        ttft = float(ttft_match.group(1)) if ttft_match else 0

        # Extract prefill speed
        prefill_match = re.search(r'prompt eval time\s*=.*\(\s*([\d.]+)\s*tokens per second', output)
        prefill_tps = float(prefill_match.group(1)) if prefill_match else 0

        # Extract decode speed
        decode_match = re.search(r'eval time\s*=.*\(\s*([\d.]+)\s*tokens per second', output)
        decode_tps = float(decode_match.group(1)) if decode_match else 0

        # Extract eval time for decode latency
        eval_time_match = re.search(r'eval time\s*=\s*([\d.]+)\s*ms', output)
        eval_time = float(eval_time_match.group(1)) if eval_time_match else 0

        return {
            'size_gb': size_gb,
            'ttft_ms': ttft,
            'prefill_tps': prefill_tps,
            'decode_tps': decode_tps,
            'prefill_lat_ms': ttft,
            'decode_lat_ms': eval_time
        }
    except Exception as e:
        print(f"Error: {e}")
        return None


In [ ]:

# ============================================================================
# PART 3: Run Profiling on All Models
# ============================================================================

print("\n" + "=" * 70)
print("PART 2: PERFORMANCE PROFILING")
print("=" * 70)

models_to_profile = [
    ("Pruned Q4_0", "/content/drive/MyDrive/LLM_Compression_Project_final_hopefully/llama2_pruned_q4_0.gguf"),
    ("Pruned Q4_K_M", "/content/drive/MyDrive/LLM_Compression_Project_final_hopefully/llama2_pruned_q4_k_m.gguf"),
    ("Pruned Q8_0", "/content/drive/MyDrive/LLM_Compression_Project_final_hopefully/llama2_pruned_q8_0.gguf"),
    ("SmoothQuant Fixed Q4_0", "/content/llama2_smoothquant_fixed_q4_0.gguf"),
    ("SmoothQuant Fixed Q4_K_M", "/content/llama2_smoothquant_fixed_q4_k_m.gguf"),
    ("SmoothQuant Fixed Q8_0", "/content/llama2_smoothquant_fixed_q8_0.gguf"),
]

profiling_results = {}

print("\nRunning profiling (this may take a few minutes)...\n")

for name, path in models_to_profile:
    if os.path.exists(path):
        print(f"  Profiling: {name}...", end=" ", flush=True)
        metrics = profile_model(path)
        if metrics:
            profiling_results[name] = metrics
            print(f"✓ (TTFT: {metrics['ttft_ms']:.1f}ms, Decode: {metrics['decode_tps']:.1f} t/s)")
        else:
            print("✗ Failed")
    else:
        print(f"  Skipping: {name} (not found)")

# Display results table
print("\n" + "=" * 100)
print("PROFILING RESULTS TABLE")
print("=" * 100)

print(f"\n{'Model':<28} {'Size':>8} {'TTFT':>10} {'Prefill':>12} {'Decode':>10} {'Prefill Lat':>12} {'Decode Lat':>12}")
print(f"{'':28} {'(GB)':>8} {'(ms)':>10} {'(tok/s)':>12} {'(tok/s)':>10} {'(ms)':>12} {'(ms)':>12}")
print("-" * 100)

for name, metrics in profiling_results.items():
    print(f"{name:<28} {metrics['size_gb']:>8.2f} {metrics['ttft_ms']:>10.1f} "
          f"{metrics['prefill_tps']:>12.1f} {metrics['decode_tps']:>10.1f} "
          f"{metrics['prefill_lat_ms']:>12.1f} {metrics['decode_lat_ms']:>12.1f}")




PART 2: PERFORMANCE PROFILING

Running profiling (this may take a few minutes)...

  Profiling: Pruned Q4_0... ✓ (TTFT: 393.5ms, Decode: 0.0 t/s)
  Profiling: Pruned Q4_K_M... ✓ (TTFT: 417.9ms, Decode: 0.0 t/s)
  Profiling: Pruned Q8_0... ✓ (TTFT: 424.1ms, Decode: 0.0 t/s)
  Profiling: SmoothQuant Fixed Q4_0... ✓ (TTFT: 382.5ms, Decode: 0.0 t/s)
  Profiling: SmoothQuant Fixed Q4_K_M... ✓ (TTFT: 433.7ms, Decode: 0.0 t/s)
  Profiling: SmoothQuant Fixed Q8_0... ✓ (TTFT: 681.9ms, Decode: 0.0 t/s)

PROFILING RESULTS TABLE

Model                            Size       TTFT      Prefill     Decode  Prefill Lat   Decode Lat
                                 (GB)       (ms)      (tok/s)    (tok/s)         (ms)         (ms)
----------------------------------------------------------------------------------------------------
Pruned Q4_0                      3.09      393.5          0.0        0.0        393.5        393.5
Pruned Q4_K_M                    3.29      417.9          0.0        0.0     

In [ ]:
# ============================================================================
# PART 4: Inference Quality Test
# ============================================================================

print("\n" + "=" * 70)
print("PART 3: INFERENCE QUALITY TEST")
print("=" * 70)

test_prompts = [
    "The capital of France is",
    "Artificial intelligence is",
    "The largest planet in our solar system is"
]

models_to_test = [
    ("Pruned Q4_0", "/content/drive/MyDrive/LLM_Compression_Project_final_hopefully/llama2_pruned_q4_0.gguf"),
    ("SmoothQuant Fixed Q4_K_M", "/content/drive/MyDrive/LLM_Compression_Project_final_hopefully/llama2_smoothquant_fixed_q4_k_m.gguf"),
]

for model_name, model_path in models_to_test:
    if not os.path.exists(model_path):
        continue

    print(f"\n{'='*60}")
    print(f"Model: {model_name}")
    print(f"{'='*60}")

    for prompt in test_prompts:
        cmd = [
            "./llama.cpp/build/bin/llama-completion",
            "-m", model_path,
            "-p", prompt,
            "-n", "40",
            "--temp", "0.7"
        ]

        try:
            result = subprocess.run(cmd, capture_output=True, text=True, timeout=120)
            output = result.stdout

            # Extract generated text
            if prompt in output:
                generated = output[output.find(prompt):].strip()
                # Clean up and limit length
                generated = generated.split('\n')[0][:200]
            else:
                generated = output[:200]

            print(f"\nPrompt: {prompt}")
            print(f"Output: {generated}")
        except Exception as e:
            print(f"\nPrompt: {prompt}")
            print(f"Error: {e}")



PART 3: INFERENCE QUALITY TEST

Model: Pruned Q4_0

Prompt: The capital of France is
Output: The capital of France is the 11th most populous city in the world, with a population of 2. kwiet 2018, The population of the French capital is 12,45

Prompt: Artificial intelligence is
Output: Artificial intelligence is a field of science and technology that studies the computational modeling of the structure and function of the brain and the behavior of the entire organism. Unterscheidung 

Prompt: The largest planet in our solar system is
Output: The largest planet in our solar system is a ball of water, and it’s not too different from what’s happening on Earth, either. everybody.

Model: SmoothQuant Fixed Q4_K_M

Prompt: The capital of France is
Output: The capital of France is the second-most visited city in the world. In 2015, 19.1 million visitors came to the city. Paris is a city of history, art, fashion,

Prompt: Artificial intelligence is
Output: Artificial intelligence is the biggest

# Deployment on CPU using Gradio Interface.

### i have not saved the .cpp files in the drive but below cell can be tested by running the Llama.cpp cell up in the code and running the below cell code. The Pi Device Deployment is shared in a Picture in the folder shared.

In [ ]:
!pip install gradio -q

import gradio as gr
import subprocess

MODEL_PATH = "/content/drive/MyDrive/Group-11/llama2_smoothquant_fixed_q4_k_m.gguf"

def generate_response(prompt, max_tokens=100):
    cmd = [
        "./llama.cpp/build/bin/llama-completion",
        "-m", MODEL_PATH,
        "-p", prompt,
        "-n", str(max_tokens),
        "--temp", "0.7"
    ]
    result = subprocess.run(cmd, capture_output=True, text=True, timeout=120)
    return result.stdout.strip()

demo = gr.Interface(
    fn=generate_response,
    inputs=[
        gr.Textbox(label="Enter your prompt", placeholder="The capital of France is..."),
        gr.Slider(minimum=20, maximum=200, value=50, label="Max Tokens")
    ],
    outputs=gr.Textbox(label="Generated Response"),
    title="LLaMA-2-7B Compressed Model Demo",
    description="75.6% compressed model (3.29 GB) running on CPU"
)

demo.launch(share=True)  # share=True gives you a public URL

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://ae99e98dce7754e2a0.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


# Thanks for reviewing our Project Code. hoping for the Best

# Sigining out with Allah Hafiz.